# Superstore Profitability Analysis
**Tools:** SQL Server · Python · Pandas · Matplotlib · Seaborn  
**Dataset:** 9,994 transactions | 2014–2017 | Sample Superstore (Tableau Public)  
**GitHub:** https://github.com/Karuna9502/superstore-profitability-analysis

---
## Business Problem
A retail superstore wanted to understand which products, regions, and customer segments 
are profitable — and which are destroying value. Nearly 1 in 5 orders was loss-making 
and management did not know why.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Working directory ──────────────────────────────────────────
os.chdir(r'C:\Users\Karuna\OneDrive\Desktop\Sample_store_project_sql')
os.makedirs('outputs', exist_ok=True)

# ── Chart styling ──────────────────────────────────────────────
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.dpi'      : 150,
    'axes.titlesize'  : 14,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
})

COLORS = {
    'profit' : '#2ecc71',
    'loss'   : '#e74c3c',
    'neutral': '#3498db',
    'warn'   : '#f39c12',
}

print("Setup complete")
print("Working directory:", os.getcwd())
print("Outputs folder ready:", os.path.exists('outputs'))


## Step 1 — Load & Validate Data

In [ ]:
# Load data — headers exist, dates in YYYY-MM-DD format
df = pd.read_csv('superstore_clean.csv',
                 parse_dates=['Order_Date', 'Ship_Date'])

df['Profit']            = df['Profit'].fillna(0)
df['Profit_Margin_Pct'] = df['Profit_Margin_Pct'].fillna(0)

print(f"Rows loaded    : {len(df):,}")
print(f"Columns        : {df.shape[1]}")
print(f"Date range     : {df['Order_Date'].min().date()} to {df['Order_Date'].max().date()}")
print(f"Nulls remaining: {df.isnull().sum().sum()}")
df.head()


## Step 2 — Data Quality Summary

| Issue | Finding | Resolution |
|---|---|---|
| NULL values | 1 NULL in Profit column | Set to 0 |
| Duplicates | 8 duplicate Order+Product combinations | Investigated, retained |
| Discount range | Max discount 80% — 856 orders above 50% | Flagged as business risk |
| Loss-making orders | 1,870 rows with negative profit | Core focus of analysis |


## Analysis 1 — Sub-Category Profitability
**Business Question:** Which product sub-categories are making money and which are losing money?


In [ ]:
subcat = (df.groupby('Sub_Category')
            .agg(total_profit=('Profit', 'sum'))
            .reset_index()
            .sort_values('total_profit'))

fig, ax = plt.subplots(figsize=(10, 8))
colors = [COLORS['loss'] if x < 0 else COLORS['profit']
          for x in subcat['total_profit']]
bars = ax.barh(subcat['Sub_Category'], subcat['total_profit'], color=colors)

for bar, value in zip(bars, subcat['total_profit']):
    offset = subcat['total_profit'].abs().max() * 0.02
    label_x = value + offset if value >= 0 else value - offset
    ha = 'left' if value >= 0 else 'right'
    ax.text(label_x, bar.get_y() + bar.get_height()/2,
            f'${value:,.0f}', va='center', ha=ha, fontsize=9)

ax.set_xlim(subcat['total_profit'].min() * 1.25,
            subcat['total_profit'].max() * 1.15)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Profit ($)')
ax.set_title('Profitability by Sub-Category\nFurniture sub-categories are the primary loss drivers',
             loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/chart1_subcategory_profit.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 1 saved")


**Finding:** Tables lost $17,305 despite generating $207K in sales. Bookcases lost $3,473. 
Both are Furniture sub-categories with average discounts above 20%. Meanwhile Copiers 
generated $55,618 profit at 37% margin. The problem is not demand — it is pricing and discounting.


## Analysis 2 — Discount vs Profit Distribution
**Business Question:** Does higher discount always mean lower profit?


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
discount_order = sorted(df['Discount'].unique())

sns.boxplot(data=df, x='Discount', y='Profit',
            order=discount_order, ax=ax)

ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit ($)')
ax.set_title('Profit Distribution by Discount Level\nDoes higher discount mean lower profit?')
ax.set_xticklabels([f'{x:.0%}' for x in discount_order], rotation=45)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/chart2_discount_vs_profit.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 2 saved")


**Finding:** The median profit line drops below $0 as discount tiers increase. 
At 50%+ discount, the entire profit distribution sits below zero — meaning the 
majority of heavily discounted orders are loss-making. Discount tiers are discrete 
(preset levels), confirming this is a deliberate pricing policy, not random variation.


## Analysis 3 — Regional Performance
**Business Question:** Which regions are most and least profitable — and why?


In [ ]:
region = (df.groupby('Region')
            .agg(total_profit=('Profit', 'sum'))
            .reset_index()
            .sort_values('total_profit', ascending=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(region['Region'], region['total_profit'], color=COLORS['neutral'])
for bar, value in zip(bars, region['total_profit']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 500,
            f'${value:,.0f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Region')
ax.set_ylabel('Total Profit ($)')
ax.set_title('Total Profit by Region\nCentral region significantly underperforms')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/chart3_regional_profit.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 3 saved")


**Finding:** Central region earns only $39,706 — nearly 3x less than West's $108,418. 
Central has more orders than South yet earns less profit. SQL analysis revealed Central 
averages 24% discount vs West's 10.9%. This is not a market problem — it is a discount 
authorisation and management control problem.


## Analysis 4 — Yearly Sales vs Profit Trend
**Business Question:** Is the business growing profitably or just growing in volume?


In [ ]:
yearly = (df.groupby('Order_Year')
            .agg(total_sales=('Sales', 'sum'),
                 total_profit=('Profit', 'sum'))
            .reset_index())
yearly['profit_margin'] = (yearly['total_profit'] /
                            yearly['total_sales'] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(yearly['Order_Year'], yearly['total_sales'],
        marker='o', label='Total Sales', color=COLORS['neutral'], linewidth=2)
ax.plot(yearly['Order_Year'], yearly['total_profit'],
        marker='o', label='Total Profit', color=COLORS['profit'], linewidth=2)

for i, row in yearly.iterrows():
    ax.annotate(f"{row['profit_margin']}% margin",
                xy=(row['Order_Year'], row['total_profit']),
                xytext=(0, -25), textcoords='offset points',
                ha='center', fontsize=8, color='darkgreen')

ax.set_xticks([2014, 2015, 2016, 2017])
ax.set_xticklabels(['2014', '2015', '2016', '2017'])
ax.set_xlabel('Year')
ax.set_ylabel('Amount ($)')
ax.set_title('Sales vs Profit Trend 2014-2017\nRevenue growing but profit margin peaked in 2016')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/chart4_yearly_trend.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 4 saved")


**Finding:** Revenue grew 51% from $484K in 2014 to $733K in 2017. However profit margin 
peaked at 13.43% in 2016 and declined to 12.80% in 2017 despite record sales. 
Volume growth is masking margin compression — a critical early warning signal that 
if unchecked will result in declining profitability despite rising revenue.


## Analysis 5 — Discount Bucket Impact
**Business Question:** What is the total financial impact of each discount tier?


In [ ]:
bucket_order = ['No Discount', 'Low 1-10pct', 'Medium 11-20pct',
                'High 21-30pct', 'Very High 31-50pct', 'Extreme 50pct+']

bucket = (df.groupby('Discount_Bucket')
            .agg(total_profit=('Profit', 'sum'))
            .reindex(bucket_order)
            .reset_index())

colors = [COLORS['profit'] if x > 0 else COLORS['loss']
          for x in bucket['total_profit']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(bucket['Discount_Bucket'], bucket['total_profit'], color=colors)
for bar, value in zip(bars, bucket['total_profit']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (2000 if value >= 0 else -8000),
            f'${value:,.0f}', ha='center', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Discount Bucket')
ax.set_ylabel('Total Profit ($)')
ax.set_title('Total Profit by Discount Bucket\nOrders above 30% discount are collectively loss-making')
ax.set_xticklabels(bucket_order, rotation=30, ha='right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/chart5_discount_bucket.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 5 saved")


**Finding:** Zero-discount orders generate $320,988 profit at 34% margin. 
Orders with extreme discounts (50%+) destroy $76,559 at -113% margin. 
The business is simultaneously running a profit engine and a loss engine. 
Eliminating discounts above 30% would recover an estimated $134,956 annually.


## Analysis 6 — Customer Segment Comparison
**Business Question:** Which customer segment is most efficient and profitable?


In [ ]:
segment = (df.groupby('Segment')
             .agg(total_profit=('Profit', 'sum'),
                  total_sales=('Sales', 'sum'),
                  total_orders=('Order_ID', 'count'))
             .reset_index())
segment['profit_margin'] = (segment['total_profit'] /
                             segment['total_sales'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(segment['Segment'], segment['total_profit'], color=COLORS['neutral'])
for i, (val, margin) in enumerate(zip(segment['total_profit'],
                                       segment['profit_margin'])):
    axes[0].text(i, val + 1500, f'${val:,.0f}\n({margin}% margin)',
                 ha='center', fontsize=9)
axes[0].set_title('Total Profit by Segment')
axes[0].set_ylabel('Total Profit ($)')
axes[0].set_ylim(0, segment['total_profit'].max() * 1.2)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].bar(segment['Segment'], segment['total_orders'], color=COLORS['warn'])
for i, val in enumerate(segment['total_orders']):
    axes[1].text(i, val + 30, f'{val:,}', ha='center', fontsize=9)
axes[1].set_title('Total Orders by Segment')
axes[1].set_ylabel('Number of Orders')
axes[1].set_ylim(0, segment['total_orders'].max() * 1.15)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Customer Segment: Home Office has fewest orders but highest margin',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/chart6_segment_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart 6 saved")


**Finding:** Consumer segment has the most orders (5,191) and highest total profit ($134,119) 
but the lowest margin at 11.55%. Home Office has the fewest orders (1,783) but the highest 
margin at 14.03%. The business should not blindly chase Consumer volume — improving Consumer 
margins through discount discipline would have the largest absolute impact.


## Analysis 7 — Correlation Analysis
**Business Question:** How strongly does discount rate predict profit margin?


In [ ]:
corr_profit = round(df['Discount'].corr(df['Profit']), 4)
corr_margin = round(df['Discount'].corr(df['Profit_Margin_Pct']), 4)

print("=" * 50)
print("KEY ANALYTICAL FINDING")
print("=" * 50)
print(f"Discount vs Profit (absolute):  {corr_profit}  (weak)")
print(f"Discount vs Profit Margin:      {corr_margin}  (very strong)")
print()
print("Interpretation:")
print("A -0.86 correlation means discount policy explains")
print("86% of profit margin variation in this business.")
print("This is not noise — it is the root cause.")


**Finding:** The -0.86 correlation between discount rate and profit margin is the single 
most powerful quantitative finding in this project. It proves — not just suggests — that 
discount policy is the primary controllable driver of profitability. Every chart in this 
analysis is supported by this number.


## Conclusions & Business Recommendations

### Summary
Analysis of 9,994 superstore transactions (2014-2017) revealed that discounting policy 
is the primary driver of profitability losses, with a -0.86 correlation between discount 
rate and profit margin.

### Key Findings
1. **18.7% of all transactions** (1,870 orders) are loss-making — totalling -$155,711
2. **Tables sub-category** lost $17,305 despite $207K in revenue (26% avg discount)
3. **Central region** averages 24% discount vs West's 10.9% — lowest margin at 7.92%
4. **Orders with 50%+ discount** destroy $76,559 collectively at -113% margin
5. **Profit margin peaked in 2016** at 13.43% and declined in 2017 — warning signal

### Business Recommendations

**1. Cap discounts at 20%**  
Require VP approval for anything above 30%.  
Estimated annual profit recovery: ~$134,956

**2. Audit Furniture category pricing**  
Tables and Bookcases are loss-making despite strong sales volumes.  
Review cost structure and implement minimum price floors.

**3. Investigate Central region discount practices**  
Average 24% discount is double the West region.  
This is a sales management control problem, not a market problem.

**4. Scale high-margin products**  
Copiers (37.2% margin), Paper (43.4% margin), Accessories (25.1% margin)  
should never receive heavy discounts — these are the profit engine.

**5. Target Home Office segment**  
Despite lowest order volume, Home Office has the highest margin (14.03%).  
Targeted marketing here improves portfolio margins without discount changes.


In [ ]:
import os
print("=" * 40)
print("ALL CHARTS SAVED TO outputs/ FOLDER")
print("=" * 40)
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}') // 1024
    print(f"  ✓ {f} ({size} KB)")
